## Generation: The retrieved document chunks and the user's question are provided to the LLM, which generates a context-aware natural language answer.


In [9]:
%run ./01_basic_rag.ipynb
%run ./02_indexing_rag.ipynb
%run ./03_retrieval_rag.ipynb

1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
384
384
Cosine Similarity: 0.7378822383225558
1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
1.3.13
Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat? I'm here to assist you with any questions or topics you'd like to discuss.
384
384
Cosine Similarity: 0.7378822383225558
 this is a bit different from self-reflection above)

Prompt LM with 100 most recent observations and to generate 3 most salient high-level questions given a set of observations/statements. Th

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

# Prompt Template banayeko, jasma retrieved context ra user ko question pass huncha
template = """
Answer the question based only on the following context.

Context:
{context}

Question:
{question}
"""

# Prompt object create gareko
prompt = ChatPromptTemplate.from_template(template)

# Local Ollama LLM load gareko
llm = ChatOllama(
    model="llama3:latest",
    temperature=0,
)

prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the following context.\n\nContext:\n{context}\n\nQuestion:\n{question}\n'), additional_kwargs={})])

In [14]:
llm = ChatOllama(model="llama3:latest", temperature=0)

In [15]:
chain = prompt | llm

In [16]:
chain.invoke({"context": docs, "question": "What is Task Decomposition?"})

AIMessage(content='Based on the provided context, Task Decomposition is not explicitly mentioned. However, it can be inferred that Task Decomposition might be related to the planning and reacting process described in the text.\n\nThe text mentions "Planning & Reacting: translate the reflections and the environment information into actions" which suggests that the agent needs to break down its goals or tasks into smaller, manageable sub-tasks (Task Decomposition) before taking action.', additional_kwargs={}, response_metadata={'model': 'llama3:latest', 'created_at': '2026-07-16T16:16:28.0133245Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11619188200, 'load_duration': 215150800, 'prompt_eval_count': 312, 'prompt_eval_duration': 821205000, 'eval_count': 87, 'eval_duration': 10576197000, 'logprobs': None, 'model_name': 'llama3:latest', 'model_provider': 'ollama'}, id='lc_run--019f6bb6-e609-7003-aa77-58ea4b169e77-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input

## Replacing LangChain Hub

`langchainhub.pull()` is no longer supported in the latest LangChain. It has been replaced with a local `ChatPromptTemplate`, making the RAG pipeline fully compatible with the latest LangChain APIs without relying on LangChain Hub.


In [24]:
from langchain_core.prompts import ChatPromptTemplate

# LangChain Hub latest version ma support nagarne bhayeko le
# local prompt create gareko ra prompt_hub_rag variable ma store gareko
prompt_hub_rag = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.

Use the following retrieved context to answer the question.

If you don't know the answer, just say you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

In [25]:
prompt_hub_rag

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks.\n\nUse the following retrieved context to answer the question.\n\nIf you don't know the answer, just say you don't know.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n"), additional_kwargs={})])

In [28]:
# RAG chain create gareko, jasma Retrieval → Prompt → LLM → Output Parser ko workflow define gareko
rag_chain = (
    {
        # Retriever bata relevant documents fetch garera string format ma context banaune
        "context": retriever | format_docs,
        # User ko question direct prompt ma pass garne
        "question": RunnablePassthrough(),
    }
    # Retrieved context ra user ko question prompt template ma inject garne
    | prompt_hub_rag
    # Prompt lai local Ollama LLM ma pathaune
    | llm
    # LLM ko response lai plain text ma convert garne
    | StrOutputParser()
)

# User ko question RAG pipeline ma pathaune
response = rag_chain.invoke("full form of RAG")

# LLM le generate gareko final answer print garne
print(response)

I don't know the answer to that question. The context provided is about writing a Super Mario game in Python with MVC components and keyboard control, but it doesn't mention anything about the full form of RAG. To provide an accurate answer, I would need more information or clarification on what RAG refers to in this context.
